In [1]:
# =========================================================
# 03. END-TO-END MODEL TRAINING & BENCHMARKING PIPELINE
# =========================================================

import os
import json
import joblib
import pandas as pd
import numpy as np
from datetime import datetime

from sklearn.model_selection import train_test_split
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

os.makedirs('models', exist_ok=True)

# ---------------------------------------------------------
# 1. LOAD DATASET WITH FLEXIBLE PATH RESOLUTION
# ---------------------------------------------------------
csv_filename = 'data_ecommerce_customer_churn.csv'

candidate_paths = [
    os.path.join('data', csv_filename),
    csv_filename,
    os.path.join('..', 'data', csv_filename),
    os.path.join('..', csv_filename)
]

csv_path = next((p for p in candidate_paths if os.path.exists(p)), None)

if csv_path is None:
    raise FileNotFoundError(f"Could not locate {csv_filename} in candidate paths.")

df = pd.read_csv(csv_path)

if 'CustomerID' in df.columns:
    df = df.drop(columns=['CustomerID'])

required_cols = [
    'Tenure', 'WarehouseToHome', 'NumberOfDeviceRegistered',
    'PreferedOrderCat', 'SatisfactionScore', 'MaritalStatus',
    'NumberOfAddress', 'Complain', 'DaySinceLastOrder',
    'CashbackAmount', 'Churn'
]

df = df[[c for c in required_cols if c in df.columns]]

X = df.drop(columns=['Churn'])
y = df['Churn']

RANDOM_STATE = 42
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

# ---------------------------------------------------------
# 2. CUSTOM DOMAIN FEATURE ENGINEER TRANSFORMER
# ---------------------------------------------------------
class EcommerceFeatureEngineer(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        if 'PreferedOrderCat' in X.columns:
            X['PreferedOrderCat'] = X['PreferedOrderCat'].replace({'Mobile': 'Mobile Phone'})

        tenure_safe = X['Tenure'].fillna(1.0)
        X['CashbackPerTenure'] = X['CashbackAmount'] / (tenure_safe + 1.0)

        days_since_safe = X['DaySinceLastOrder'].fillna(0.0)
        X['InactivityRatio'] = days_since_safe / (tenure_safe * 30.0 + 1.0)

        X['HighRiskComplain'] = (
            (X['Complain'] == 1) & (X['SatisfactionScore'] <= 2)
        ).astype(int)

        X['TenureStage'] = pd.cut(
            tenure_safe,
            bins=[-np.inf, 3, 12, 24, np.inf],
            labels=['Onboarding', 'Early', 'Established', 'Loyal']
        ).astype(str)

        return X

# ---------------------------------------------------------
# 3. DEFINE FEATURE STRUCTURE & COLUMN TRANSFORMER
# ---------------------------------------------------------
raw_num_features = [
    'Tenure', 'WarehouseToHome', 'NumberOfDeviceRegistered',
    'SatisfactionScore', 'NumberOfAddress', 'Complain',
    'DaySinceLastOrder', 'CashbackAmount'
]

raw_cat_features = ['PreferedOrderCat', 'MaritalStatus']
engineered_num_features = ['CashbackPerTenure', 'InactivityRatio', 'HighRiskComplain']
engineered_cat_features = ['TenureStage']

full_num_features = raw_num_features + engineered_num_features
full_cat_features = raw_cat_features + engineered_cat_features

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_pipeline, full_num_features),
        ('cat', cat_pipeline, full_cat_features)
    ]
)

# ---------------------------------------------------------
# 4. BENCHMARK CANDIDATE CLASSIFIERS
# ---------------------------------------------------------
models = {
    'Logistic Regression': LogisticRegression(class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE),
    'Decision Tree': DecisionTreeClassifier(random_state=RANDOM_STATE),
    'Random Forest': RandomForestClassifier(n_estimators=300, max_depth=20, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1),
    'XGBoost': XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05, random_state=RANDOM_STATE, eval_metric='logloss')
}

comparison_rows = []

for name, clf in models.items():
    pipe = Pipeline([
        ('feature_engineer', EcommerceFeatureEngineer()),
        ('preprocessor', preprocessor),
        ('classifier', clf)
    ])
    
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    y_prob = pipe.predict_proba(X_test)[:, 1]
    
    comparison_rows.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall': recall_score(y_test, y_pred, zero_division=0),
        'F1 Score': f1_score(y_test, y_pred, zero_division=0),
        'ROC-AUC': roc_auc_score(y_test, y_prob)
    })

comparison_df = pd.DataFrame(comparison_rows).sort_values(by='F1 Score', ascending=False)
print("\n=== Candidate Model Benchmarks ===")
print(comparison_df.to_string(index=False))

# Save baseline model comparison table
comparison_df.to_csv('models/model_comparison.csv', index=False)

# ---------------------------------------------------------
# 5. SAVE INITIAL PRODUCTION MODEL PIPELINE
# ---------------------------------------------------------
selected_classifier = RandomForestClassifier(
    n_estimators=300,
    max_depth=20,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features='sqrt',
    class_weight='balanced',
    random_state=RANDOM_STATE,
    n_jobs=-1
)

production_pipeline = Pipeline([
    ('feature_engineer', EcommerceFeatureEngineer()),
    ('preprocessor', preprocessor),
    ('classifier', selected_classifier)
])

production_pipeline.fit(X_train, y_train)

joblib.dump(production_pipeline, 'models/churn_model.joblib')

OPTIMAL_THRESHOLD = 0.4500
joblib.dump(OPTIMAL_THRESHOLD, 'models/churn_threshold.joblib')

metadata = {
    "model_name": "Random Forest Classifier (End-to-End Production Pipeline)",
    "model_version": "1.0.0",
    "training_date": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "target_name": "Churn",
    "optimal_threshold": OPTIMAL_THRESHOLD,
    "raw_input_features": X_train.columns.tolist(),
    "raw_numerical_features": raw_num_features,
    "raw_categorical_features": raw_cat_features,
    "engineered_features": engineered_num_features + engineered_cat_features,
    "pipeline_steps": ["feature_engineer", "preprocessor", "classifier"]
}

with open('models/pipeline_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=4)

print("\nModel training outputs successfully saved to models/:")
print(" - models/model_comparison.csv")
print(" - models/churn_model.joblib")
print(" - models/churn_threshold.joblib")
print(" - models/pipeline_metadata.json")


=== Candidate Model Benchmarks ===
              Model  Accuracy  Precision   Recall  F1 Score  ROC-AUC
            XGBoost  0.936629    0.85124 0.762963  0.804688 0.957968
      Decision Tree  0.935361    0.83871 0.770370  0.803089 0.869895
      Random Forest  0.930292    0.78169 0.822222  0.801444 0.958761
Logistic Regression  0.830165    0.50237 0.785185  0.612717 0.898086



Model training outputs successfully saved to models/:
 - models/model_comparison.csv
 - models/churn_model.joblib
 - models/churn_threshold.joblib
 - models/pipeline_metadata.json
